In [ ]:
!pip install -q x-transformers
!pip install -q flash-attn --no-build-isolation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import os
import sys
import subprocess
import hashlib
import gc
from datetime import datetime
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import RobertaTokenizerFast, get_cosine_schedule_with_warmup, DataCollatorForLanguageModeling
from datasets import load_dataset
from x_transformers import Encoder

# ==========================================
# 1. CONFIGURATION
# ==========================================
# YOUR REPO ID (Created in previous step)
HF_ID = "prism-lab/wikitext-103-prism-32k-seq4k"

# Hyperparameters
VOCAB_SIZE = 32768
SEQ_LEN = 4096
BATCH_SIZE = 8
EPOCHS = 40
LR = 1e-3
D_MODEL = 512
D_BRANCH = 256
DEPTH = 9
RESUME_PATH = None #"/content/drive/MyDrive/PRISM_Experiments/PILLARS_SplitStream_8Layer_20260116_025321_8438ce62/last.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# ==========================================
# 2. DATA PIPELINE (The "Pro" Way)
# ==========================================
def prepare_data_from_hub():
    print(f"⬇️ Pulling Pre-Tokenized Data from {HF_ID}...")

    # 1. Load Tokenizer (Instant)
    # This pulls the exact tokenizer you uploaded
    tokenizer = RobertaTokenizerFast.from_pretrained(HF_ID)

    # 2. Load Dataset (Instant)
    # This pulls the already chunked/tokenized data
    dataset = load_dataset(HF_ID)

    print(f"✅ Loaded {len(dataset['train'])} training chunks.")

    # 3. Collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    return dataset, data_collator
# ==========================================
# 3. PRISM ARCHITECTURE (Complex-Valued)
# ==========================================

class ComplexDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
    def forward(self, z):
        if not self.training or self.p == 0.0: return z
        mask = torch.ones_like(z.real)
        mask = F.dropout(mask, self.p, self.training, inplace=False)
        return z * mask

class RobustPhaseNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        mag = torch.abs(x)
        rms = torch.sqrt(torch.mean(mag**2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

class ModReLU(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.b = nn.Parameter(torch.zeros(features))
    def forward(self, z):
        mag = torch.abs(z)
        new_mag = F.relu(mag + self.b)
        phase = z / (mag + 1e-6)
        return new_mag * phase

class ComplexToRealBridge(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x_complex):
        cat = torch.cat([x_complex.real, x_complex.imag], dim=-1)
        return self.norm(self.proj(cat))

# ==========================================
# 4. DYNAMIC RoSE (Mamba-3 Engine)
# ==========================================
class DynamicRoSE(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, max_period=10000.0):
        super().__init__()
        self.embedding_dim = embedding_dim

        # 1. Master Real Embedding (The "Particle")
        self.raw_embedding = nn.Embedding(num_embeddings, embedding_dim)

        # 2. Complex Adapter (The "Wave" Magnitude/Initial Phase)
        self.adapter = nn.Linear(embedding_dim, embedding_dim * 2)

        # 3. Static Frequencies (Positional)
        freqs = torch.exp(torch.arange(0, embedding_dim, dtype=torch.float32) * -(math.log(max_period) / embedding_dim))
        self.register_buffer('freqs', freqs)

        self.rotation_predictor = nn.Linear(embedding_dim, embedding_dim * 2)

    def forward(self, input_ids):
        # A. Raw Particle
        real_base = self.raw_embedding(input_ids)
        B, L, D = real_base.shape

        # B. Complex Wave Content
        complex_params = self.adapter(real_base)
        z_t = torch.complex(complex_params[..., :D], complex_params[..., D:])

        rot_raw = self.rotation_predictor(real_base)
        rot_x, rot_y = rot_raw.chunk(2, dim=-1)

        rot_mag = torch.sqrt(rot_x**2 + rot_y**2 + 1e-6)
        dynamic_rot = torch.complex(rot_x / rot_mag, rot_y / rot_mag)

        # D. Static Positional Rotation
        pos = torch.arange(L, device=input_ids.device).float()
        static_angles = torch.outer(pos, self.freqs) # [L, D]
        static_rot = torch.polar(torch.ones_like(static_angles), static_angles) # [L, D]

        z_final = z_t * static_rot.unsqueeze(0) * dynamic_rot

        return z_final, real_base

# ==========================================
# 5. HYENA FILTER
# ==========================================
class HyenaNeuralFilter(nn.Module):
    def __init__(self, d_model, max_len=1024, hidden_dim=64):
        super().__init__()
        self.d_model = d_model
        freqs = torch.exp(torch.arange(0, hidden_dim, 2, dtype=torch.float32) * -(math.log(10000.0) / hidden_dim))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, d_model * 2)
        )
    def forward(self, L, device):
        t = torch.linspace(0, 1, steps=L, device=device).unsqueeze(-1)
        emb = torch.cat([torch.sin(t * self.freqs), torch.cos(t * self.freqs)], dim=-1)
        out = self.mlp(emb).view(L, self.d_model, 2)
        return torch.complex(out[..., 0], out[..., 1])

# ==========================================
# 6. GATED HARMONIC CONVOLUTION (Lean)
# ==========================================
class GatedHarmonicConvolution(nn.Module):
    def __init__(self, d_model, max_len=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.filter_len = max_len
        self.neural_filter = HyenaNeuralFilter(d_model, max_len=max_len)
        self.gate_proj = nn.Linear(d_model * 2, d_model * 2)
        self.mix_real = nn.Linear(d_model, d_model)
        self.mix_imag = nn.Linear(d_model, d_model)
        self.out_real = nn.Linear(d_model, d_model)
        self.out_imag = nn.Linear(d_model, d_model)
        self.activation = ModReLU(d_model)
        self.norm = RobustPhaseNorm(d_model)
        self.dropout = ComplexDropout(dropout)

    def forward(self, x, src_mask=None):
        residual = x
        x_norm = self.norm(x)
        if src_mask is not None:
             x_norm = x_norm.masked_fill(src_mask.unsqueeze(-1), 0.0)

        # 1. Global Beam (FFT + Hyena)
        B, L, D = x_norm.shape
        eff_L = min(L, self.filter_len)
        x_freq = torch.fft.fft(x_norm, n=eff_L, dim=1, norm='ortho')
        h = self.neural_filter(eff_L, x.device).unsqueeze(0)
        x_filtered = x_freq * h
        x_time = torch.fft.ifft(x_filtered, n=eff_L, dim=1, norm='ortho')
        if L > eff_L: x_time = F.pad(x_time, (0,0,0,L-eff_L))
        else: x_time = x_time[:, :L, :]

        # 2. Gating
        gates = torch.sigmoid(self.gate_proj(torch.cat([x_norm.real, x_norm.imag], dim=-1)))
        g_r, g_i = gates.chunk(2, dim=-1)
        x_gated = torch.complex(x_time.real * g_r, x_time.imag * g_i)

        # 3. Mixing & Out
        mr, mi = self.mix_real, self.mix_imag
        x_mixed = torch.complex(mr(x_gated.real) - mi(x_gated.imag), mr(x_gated.imag) + mi(x_gated.real))
        x_act = self.activation(x_mixed)
        or_, oi = self.out_real, self.out_imag
        out = torch.complex(or_(x_act.real) - oi(x_act.imag), or_(x_act.imag) + oi(x_act.real))
        return self.dropout(out) + residual

# ==========================================
# 7. MODEL WRAPPERS
# ==========================================
class PRISMEncoder(nn.Module):
    def __init__(self, num_layers, d_model, max_len, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GatedHarmonicConvolution(d_model, max_len, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = RobustPhaseNorm(d_model)
    def forward(self, x, src_mask=None):
        for layer in self.layers:
            if self.training: x = torch.utils.checkpoint.checkpoint(layer, x, src_mask, use_reentrant=False)
            else: x = layer(x, src_mask)
        return self.final_norm(x)

class PRISM_WikiText_Model(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, prism_depth=5, trans_depth=1, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 1. PRISM Core (The Optical/Passive Part)
        self.rose = DynamicRoSE(vocab_size, d_model)
        self.prism_encoder = PRISMEncoder(prism_depth, d_model, max_len=max_len, dropout=dropout)
        self.bridge = ComplexToRealBridge(d_model)
        self.periscope_proj = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model), nn.GELU())

        # 2. Refiner (The Digital/Active Part)
        # 🔄 SWAPPED: Replaced Standard Transformer with RoPE-Enabled Encoder
        if trans_depth > 0:
            self.refiner = Encoder(
                dim=d_model,
                depth=trans_depth,
                heads=8,
                rotary_pos_emb=True,
                attn_flash=True,
                attn_dropout=dropout,
                ff_dropout=dropout,

            )
        else:
            self.refiner = None

        # 3. Output
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.lm_head.weight = self.rose.raw_embedding.weight

    def forward(self, input_ids):
        # A. Wave Physics
        wave_src, particle_src = self.rose(input_ids)
        wave_out = self.prism_encoder(wave_src)
        wave_real = self.bridge(wave_out)

        # B. Interface
        mixed_memory = self.periscope_proj(torch.cat([wave_real, particle_src], dim=-1))

        # C. Digital Refinement (Now with RoPE)
        if self.refiner:
            out = self.refiner(mixed_memory)
        else:
            out = mixed_memory

        return self.lm_head(out)

class FNetBlock(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm_mix = nn.LayerNorm(d_model) # LayerNorm is safer for FNet than RMSNorm
        self.norm_ff = nn.LayerNorm(d_model)

        self.mix_dropout = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # 1. Fourier Mixing Branch
        residual = x
        x = self.norm_mix(x)

        # --- THE FIX ---
        with torch.cuda.amp.autocast(enabled=False):
            x = x.float()
            # norm='ortho' makes the FFT energy-preserving.
            # Output magnitude will match input magnitude (~1).
            x = torch.fft.fftn(x, dim=(-2, -1), norm='ortho').real
            x = x.to(dtype=residual.dtype)
        # ---------------

        # Now 'x' and 'residual' have roughly same magnitude.
        # The skip connection works again.
        x = self.mix_dropout(x)
        x = x + residual

        # 2. Feed Forward Branch
        residual = x
        x = self.norm_ff(x)
        x = self.ff(x)
        return x + residual


class FNetEncoder(nn.Module):
    def __init__(self, depth, d_model, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            FNetBlock(d_model, d_ff, dropout) for _ in range(depth)
        ])
        # [FIX] Use LayerNorm here to match the blocks
        self.norm_out = nn.LayerNorm(d_model)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm_out(x)

class Pillars_DualStream(nn.Module):
    def __init__(self, vocab_size, d_model=512, d_branch=384, seq_len=4096, depth=4):
        super().__init__()
        self.d_branch = d_branch
        self.d_refiner = d_model

        # --- A. Rate Stream (FNet) ---
        self.fnet_emb = nn.Embedding(vocab_size, d_branch)
        self.fnet_pos = nn.Embedding(seq_len, d_branch)
        self.stream_rate = FNetEncoder(depth=depth, d_model=d_branch, d_ff=d_branch*4, dropout=0.1)

        # --- B. Phase Stream (PRISM) ---
        self.stream_phase_emb = DynamicRoSE(vocab_size, d_branch)
        self.stream_phase = PRISMEncoder(num_layers=depth, d_model=d_branch, max_len=seq_len, dropout=0.1)
        self.phase_bridge = ComplexToRealBridge(d_branch)

        # --- C. Fusion (The Funnel) ---
        self.fusion_proj = nn.Linear(d_branch * 2, d_model)
        self.fusion_norm = nn.LayerNorm(d_model)

        # --- D. Refiner ---
        self.refiner = Encoder(
            dim=d_model, depth=1, heads=8, attn_flash=True,
            rotary_pos_emb=True, attn_dropout=0.1, ff_dropout=0.1
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # 1. Rate Path
        f_emb = self.fnet_emb(x) + self.fnet_pos(torch.arange(x.shape[1], device=x.device))
        rate_out = self.stream_rate(f_emb)

        # 2. Phase Path
        p_src, _ = self.stream_phase_emb(x)
        phase_out = self.phase_bridge(self.stream_phase(p_src))

        # 3. Fusion
        fused = self.fusion_norm(self.fusion_proj(torch.cat([rate_out, phase_out], dim=-1)))

        # 4. Refine & Output
        return self.lm_head(self.refiner(fused))


class Pillars_Compact(nn.Module):
    def __init__(self, vocab_size, d_model=512, d_branch=384, seq_len=4096, depth=4):
        super().__init__()
        self.d_model = d_model
        self.d_branch = d_branch

        # 1. SHARED ROOT
        self.rose = DynamicRoSE(vocab_size, d_model)

        # 2. DOWNSAMPLE (512 -> 384)
        self.particle_down = nn.Linear(d_model, d_branch)
        self.wave_down = nn.Linear(d_model * 2, d_branch * 2)

        # 3. RATE STREAM (FNet, Depth 4)
        self.fnet_pos = nn.Embedding(seq_len, d_branch)
        self.stream_rate = FNetEncoder(depth=depth, d_model=d_branch, d_ff=d_branch*4, dropout=0.1)

        # 4. PHASE STREAM (PRISM, Depth 4)
        self.stream_phase = PRISMEncoder(num_layers=depth, d_model=d_branch, max_len=seq_len, dropout=0.1)
        self.phase_bridge = ComplexToRealBridge(d_branch)

        # 5. FUSION (Clean Projection)
        # Input: 384 (Rate) + 384 (Phase) = 768
        # Output: 512 (Refiner Dim)
        self.fusion_proj = nn.Linear(d_branch * 2, d_model)
        self.fusion_norm = nn.LayerNorm(d_model)

        # 6. REFINER (The Brain)
        self.refiner = Encoder(
            dim=d_model, depth=1, heads=8, attn_flash=True,
            rotary_pos_emb=True, attn_dropout=0.1, ff_dropout=0.1
        )

        # 7. TIED HEAD
        self.head_bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, input_ids):
        # A. Shared Root
        wave_src, particle_src = self.rose(input_ids)

        # B. Downsample
        p_small = self.particle_down(particle_src)
        w_flat = torch.cat([wave_src.real, wave_src.imag], dim=-1)
        w_small_flat = self.wave_down(w_flat)
        w_small = torch.complex(w_small_flat[..., :self.d_branch], w_small_flat[..., self.d_branch:])

        # C. Branches
        pos_emb = self.fnet_pos(torch.arange(input_ids.shape[1], device=input_ids.device))
        rate_out = self.stream_rate(p_small + pos_emb)
        phase_out = self.phase_bridge(self.stream_phase(w_small))

        # D. Fusion (Concat -> Project)
        # We rely on the Transformer Refiner to attend to the right parts.
        stacked = torch.cat([rate_out, phase_out], dim=-1)
        context = self.fusion_norm(self.fusion_proj(stacked))

        # E. Refiner
        refined = self.refiner(context)

        # F. Output
        logits = F.linear(refined, self.rose.raw_embedding.weight, self.head_bias)

        return logits

import torch
import torch.nn as nn
from prettytable import PrettyTable # Optional, but makes tables nice.
# If you don't have prettytable, the code below uses standard f-strings.

import torch
import torch.nn as nn

import torch
import torch.nn as nn

def deep_analyze_pillars(model):
    def get_p(obj):
        """Safely returns parameter count for Modules OR raw Parameters."""
        if isinstance(obj, nn.Parameter):
            return obj.numel()
        return sum(p.numel() for p in obj.parameters() if p.requires_grad)

    def format_num(n):
        if n > 1e6: return f"{n/1e6:.2f}M"
        if n > 1e3: return f"{n/1e3:.2f}K"
        return str(n)

    print("\n" + "="*80)
    print(f"🏗️  PILLARS (COMPACT) - DEEP LAYER ANALYSIS")
    print("="*80)
    print(f"{'MODULE / LAYER':<40} | {'PARAMS':<15} | {'TYPE'}")
    print("-" * 80)

    total_params = get_p(model)

    # -----------------------------------------------
    # 1. STATIC MEMORY (Embeddings)
    # -----------------------------------------------
    vocab_emb = get_p(model.rose.raw_embedding)
    fnet_pos  = get_p(model.fnet_pos)

    print(f"{'Shared Vocab Embedding':<40} | {format_num(vocab_emb):<15} | 💾 STORAGE")
    print(f"{'FNet Positional Embedding':<40} | {format_num(fnet_pos):<15} | 💾 STORAGE")

    # -----------------------------------------------
    # 2. INPUT LOGIC (RoSE & Downsampling)
    # -----------------------------------------------
    rose_total = get_p(model.rose)
    rose_logic = rose_total - vocab_emb # Subtract the embedding matrix we already counted

    print("-" * 80)
    print(f"{'Dynamic RoSE (Adapters)':<40} | {format_num(rose_logic):<15} | 🌊 PHASE INIT")
    print(f"{'Particle Downsample (512->384)':<40} | {format_num(get_p(model.particle_down)):<15} | 📉 PROJ")
    print(f"{'Wave Downsample (1024->768)':<40} | {format_num(get_p(model.wave_down)):<15} | 📉 PROJ")

    # -----------------------------------------------
    # 3. STREAM A: RATE (FNet)
    # -----------------------------------------------
    print("-" * 80)
    print(f"TRACK A: RATE STREAM (FNet) - Depth {len(model.stream_rate.layers)}")

    fnet_encoder_total = 0
    for i, layer in enumerate(model.stream_rate.layers):
        p = get_p(layer)
        fnet_encoder_total += p
        print(f"  ├─ FNet Block {i:<24} | {format_num(p):<15} | ⚡ RATE")

    fnet_norm = get_p(model.stream_rate.norm_out)
    fnet_encoder_total += fnet_norm
    print(f"  └─ Final Norm {i:<24} | {format_num(fnet_norm):<15} | ⚡ RATE")

    # -----------------------------------------------
    # 4. STREAM B: PHASE (PRISM)
    # -----------------------------------------------
    print("-" * 80)
    print(f"TRACK B: PHASE STREAM (PRISM) - Depth {len(model.stream_phase.layers)}")

    prism_encoder_total = 0
    for i, layer in enumerate(model.stream_phase.layers):
        p = get_p(layer)
        prism_encoder_total += p
        print(f"  ├─ PRISM Block {i:<23} | {format_num(p):<15} | 🌊 PHASE")

    prism_norm = get_p(model.stream_phase.final_norm)
    prism_encoder_total += prism_norm
    print(f"  └─ Final Norm {i:<24} | {format_num(prism_norm):<15} | 🌊 PHASE")

    bridge_p = get_p(model.phase_bridge)
    print(f"{'Phase Bridge (Complex->Real)':<40} | {format_num(bridge_p):<15} | 🌉 BRIDGE")

    # -----------------------------------------------
    # 5. THE BRAIN (Fusion & Refiner)
    # -----------------------------------------------
    print("-" * 80)
    fusion_p = get_p(model.fusion_proj) + get_p(model.fusion_norm)
    print(f"{'Fusion (Concat -> Proj -> Norm)':<40} | {format_num(fusion_p):<15} | 🧠 FUSION")

    refiner_p = get_p(model.refiner)
    print(f"{'Transformer Refiner (1 Layer)':<40} | {format_num(refiner_p):<15} | 🧠 ATTENTION")

    # [FIX] Handle nn.Parameter directly
    head_bias_p = get_p(model.head_bias)
    print(f"{'Output Head Bias':<40} | {format_num(head_bias_p):<15} | 🎯 OUTPUT")

    # -----------------------------------------------
    # 6. SUMMARY
    # -----------------------------------------------
    print("="*80)

    storage = vocab_emb + fnet_pos + head_bias_p
    active = total_params - storage

    print(f"TOTAL PARAMETERS:      {total_params/1e6:.2f} M")
    print(f"   ├─ 💾 Storage:      {storage/1e6:.2f} M  (Embeddings)")
    print(f"   └─ 🧠 Compute:      {active/1e6:.2f} M  (Logic/Weights)")
    print("-" * 80)
    print(f"STREAM BREAKDOWN:")
    print(f"   ├─ ⚡ Rate Stream:   {fnet_encoder_total/1e6:.2f} M")
    print(f"   └─ 🌊 Phase Stream:  {prism_encoder_total/1e6:.2f} M")
    print("="*80 + "\n")

    return total_params

model = Pillars_Compact(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    d_branch=D_BRANCH,
    seq_len=SEQ_LEN,
    depth=DEPTH
).to(DEVICE)
deep_analyze_pillars(model)

In [ ]:

# Run the parameter analysis to confirm strict adherence to budget
def analyze_pillars_compact(model):
    print("\n" + "="*70)
    print("🏛️  PILLARS COMPACT: ARCHITECTURAL COST ANALYSIS")
    print("="*70)

    stats = {
        "Shared Memory (Storage)": 0,
        "Rate Stream (FNet)": 0,
        "Phase Stream (PRISM)": 0,
        "Fusion & Refiner": 0,
        "Tied Head Bias": 0
    }

    total_params = 0

    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        n = param.numel()
        total_params += n

        if "rose.raw_embedding" in name:
            stats["Shared Memory (Storage)"] += n
        elif "rose.adapter" in name or "rose.rotation" in name or "stream_phase" in name or "phase_bridge" in name:
            stats["Phase Stream (PRISM)"] += n
        elif "fnet_pos" in name or "stream_rate" in name:
            stats["Rate Stream (FNet)"] += n
        elif "gate" in name or "mix" in name or "refiner" in name or "down" in name or "proj" in name or "norm" in name:
            stats["Fusion & Refiner"] += n
        elif "head_bias" in name:
            stats["Tied Head Bias"] += n
        else:
            print(f"⚠️ Uncategorized: {name} ({n})")

    print(f"{'COMPONENT':<30} | {'PARAMS':<12} | {'% TOTAL':<8}")
    print("-" * 60)

    for category, count in stats.items():
        if count > 0:
            pct = (count / total_params) * 100
            print(f"{category:<30} | {count:12,} | {pct:6.1f}%")

    print("-" * 60)
    print(f"{'TOTAL PARAMETERS':<30} | {total_params:12,} | 100.0%")
    print("=" * 70)


    active_params = total_params - stats["Shared Memory (Storage)"] - stats["Tied Head Bias"]
    print(f"   1. Total Model Size:      {total_params/1e6:.1f}M")
    print(f"   2. Baseline Target:       ~32.5M")
    print(f"   3. Active Reasoning Params: {active_params/1e6:.1f}M (The actual brain)")
    print("="*70 + "\n")




In [ ]:
# ==========================================
# 4. LOGGING UTILITIES
# ==========================================
def generate_run_id():
    raw = datetime.now().strftime("%Y%m%d%H%M%S%f")
    return hashlib.md5(raw.encode()).hexdigest()[:8]

def log_environment(save_dir, run_id, config):
    log_path = os.path.join(save_dir, f"env_metadata_{run_id}.txt")
    with open(log_path, "w") as f:
        f.write(f"PRISM EXPERIMENT METADATA | Run ID: {run_id}\n{'='*50}\n")
        for k, v in config.items(): f.write(f"{k}: {v}\n")
    print(f"📝 Environment Snapshot saved to: {log_path}")

def log_metrics(save_dir, run_id, epoch, train_loss, val_loss, ppl):
    log_path = os.path.join(save_dir, f"metrics_log_{run_id}.csv")
    if not os.path.exists(log_path):
        with open(log_path, "w") as f: f.write("Timestamp,Epoch,Train_Loss,Val_Loss,Perplexity\n")
    with open(log_path, "a") as f:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"{ts},{epoch},{train_loss:.6f},{val_loss:.6f},{ppl:.6f}\n")


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, config):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_loss,
        'config': config
    }, path)

def init_pillars_weights(model):
    print("✨ APPLYING PILLARS INITIALIZATION PROTOCOL...")

    # 1. SHARED ROOT (RoSE) - MATCHING YOUR ORIGINAL LOGIC
    # Standard embedding init
    nn.init.normal_(model.rose.raw_embedding.weight, std=model.d_model ** -0.5)

    # Adapter: Orthogonal ensures clean entry to complex plane
    nn.init.orthogonal_(model.rose.adapter.weight)

    # --- THE ROSE IDENTITY TRICK (From your original code) ---
    # Start with almost zero rotation influence from content
    nn.init.normal_(model.rose.rotation_predictor.weight, std=0.01)
    with torch.no_grad():
        # Force initial vector to (1, 0) -> Angle 0, Mag 1
        # This allows the model to start with "Safe" static physics
        model.rose.rotation_predictor.bias[:model.d_model].fill_(1.0)
        model.rose.rotation_predictor.bias[model.d_model:].fill_(0.0)
    # -------------------------------------------------------

    # 2. DOWNSAMPLERS (The Split)
    # Scale gain by 1.414 (sqrt 2) to preserve energy when halving dimensions
    nn.init.orthogonal_(model.particle_down.weight, gain=1.414)
    nn.init.orthogonal_(model.wave_down.weight, gain=1.414)

    # 3. FNET BRANCH (Rate Stream)
    # Kaiming Normal (Good for GELU)
    for name, m in model.stream_rate.named_modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None: nn.init.zeros_(m.bias)

    # 4. PRISM BRANCH (Phase Stream)
    # Xavier Uniform (Good for Complex/Linear)
    for name, m in model.stream_phase.named_modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight, gain=1.0)
            if m.bias is not None: nn.init.zeros_(m.bias)
        # Initialize ModReLU bias slightly positive to avoid dead neurons
        if isinstance(m, ModReLU):
            nn.init.constant_(m.b, 0.01)

    # 5. FUSION & REFINER
    # Start neutral
    nn.init.xavier_uniform_(model.fusion_proj.weight, gain=1.0)

    for p in model.refiner.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    # 6. TIED HEAD BIAS
    nn.init.zeros_(model.head_bias)

    print("✅ INITIALIZATION COMPLETE.")

def run_wikitext_training(experiment_name="PILLARS_SplitStream_9Layer"):
    from google.colab import drive
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')

    # --- SETUP DIRS ---
    if RESUME_PATH and os.path.exists(RESUME_PATH):
        print(f"🔄 RESUMING FROM: {RESUME_PATH}")
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE)
        SAVE_DIR = os.path.dirname(RESUME_PATH)
        run_id = checkpoint.get('config', {}).get('run_id', 'resumed')
    else:
        run_id = hashlib.md5(datetime.now().strftime("%Y%m%d%H%M%S%f").encode()).hexdigest()[:8]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        folder_name = f"{experiment_name}_{timestamp}_{run_id}"
        SAVE_DIR = os.path.join("/content/drive/My Drive/PRISM_Experiments", folder_name)
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"💾 Checkpoints: {SAVE_DIR}")

    writer = SummaryWriter(log_dir=SAVE_DIR)
    GRAD_ACCUM = 4

    lm_datasets, data_collator = prepare_data_from_hub()

    train_loader = DataLoader(
        lm_datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=data_collator, num_workers=2, pin_memory=True,
        prefetch_factor=2, persistent_workers=True
    )
    valid_loader = DataLoader(
        lm_datasets["validation"], batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=2, pin_memory=True
    )
    test_loader = DataLoader(
        lm_datasets["test"], batch_size=BATCH_SIZE,
        collate_fn=data_collator, num_workers=2, pin_memory=True
    )

    print("\n⚡ INITIALIZING PILLARS MODEL...")

    # INSTANTIATE THE NEW MODEL
    model = Pillars_Compact(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        d_branch=D_BRANCH,
        seq_len=SEQ_LEN,
        depth=DEPTH
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01) # Added decay for stabilization
    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()

    start_epoch = 0
    best_val_loss = float('inf')

    if RESUME_PATH and os.path.exists(RESUME_PATH):
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        del checkpoint
        torch.cuda.empty_cache()
    else:
        # APPLY THE NEW INIT LOGIC
        init_pillars_weights(model)
    print(model)
    analyze_pillars_compact(model)
    print(f"\n🚀 STARTING (Ep {start_epoch+1} to {EPOCHS})")
    global_step = (len(train_loader) // GRAD_ACCUM) * start_epoch

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)

            loss = criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)) / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                # 1. Calc Norm
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                # 2. Step
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # 3. LOGGING
                actual_loss = loss.item() * GRAD_ACCUM

                # [FIX] Log Grad Norm to TensorBoard now
                writer.add_scalar('Train/Loss', actual_loss, global_step)
                writer.add_scalar('Train/GradNorm', grad_norm.item(), global_step)

                # 4. Progress Bar
                pbar.set_postfix({
                    'loss': f"{actual_loss:.4f}",
                    'gnorm': f"{grad_norm.item():.2f}"
                })

        # VALIDATION
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in valid_loader:
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                val_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()

        avg_val_loss = val_loss / len(valid_loader)
        ppl = math.exp(avg_val_loss) if avg_val_loss < 100 else float('inf')

        print(f"✨ Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | PPL: {ppl:.2f}")
        writer.add_scalar('Val/PPL', ppl, epoch+1)

        config_dump = {"epoch": epoch, "run_id": run_id}
        save_checkpoint(os.path.join(SAVE_DIR, "last.pt"), model, optimizer, scheduler, epoch, best_val_loss, config_dump)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best.pt"))
            print("   🏆 New Best Model Saved!")

    best_path = os.path.join(SAVE_DIR, "best.pt")
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path))
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Testing"):
                x, y = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
                test_loss += criterion(model(x).view(-1, VOCAB_SIZE), y.view(-1)).item()
        print(f"🏆 FINAL PPL: {math.exp(test_loss/len(test_loader)):.2f}")

    writer.close()
    return model

In [ ]:
if __name__ == "__main__":

    print("🔥 IGNITING PILLARS TRAINING PIPELINE...")

    # 1. Run the Training Routine
    # This handles Model Creation -> Analysis -> Training -> Saving
    trained_prism = run_wikitext_training()

    # 2. Cleanup & Shutdown
    print("✅ Experiment Complete. Shutting down runtime...")
    from google.colab import runtime
    runtime.unassign()

In [ ]:
from google.colab import runtime
runtime.unassign()